In [ ]:
import os
import json
import subprocess
import sys
import glob
import pandas as pd
from pathlib import Path
import src

## Heudiconv wrapper script MRE
This script runs heudiconv on dicom files, generating an overview tsv file that lists metadata for each image type (e.g. sequence that was used to generate it). <br>
Specify in / output paths for summary script in the config .json file in this folder. <br>

How to: <br>
usually, there is no need to change anything in this script. The only task you have is possibly **renaming some folders on the drive, such that they match the dicom patient id** - this will avoid confusion or double names for subjects. You will see mismatching names in the output dataframes below. This applies unless theres a naming conflict, that makes it impossible to use the patient id as foldername, more on this below. 

Requirements (already set in config file):
- *dicom root paths* - where to look for dicoms (for 3T and 7T respectively). To be specified **relative**, i.e. excluding the drive, as we might mount the harddrive differently each time.
- *output paths* - where to write the summary sheets to (for 3T and 7T respectively). To be specified **relative**, i.e. excluding the drive, as we might mount the harddrive differently each time.
- *folder templates* - how to find the dicom files (for 3T and 7T respectively). The pattern must have the placeholders for the respective dicom root path, and the subject, more on this below.

Output:
- *.heudiconv* folder containing the summaries, one for each session. Most importantly: containing the dicominfo.tsv file with the sequence infos, or a modification version thereof (for 3T and 7T respectively)

NB! 
- You need to have **the same** folder hierachy (as specified by the template patterns) for all 7T data, and 3T data, respecively. For subjects with different hierarchy heudiconv will process 0 files.

In [ ]:
def run_heudiconv_for_scanner(output_dir, pattern_template, sessions, force_heudiconv=False, dryrun = False):
    """
    Run Heudiconv without heuristic to create overview of DICOM files matching the given pattern template.

    Args:
        output_dir (str): Directory where overview sheets should be saved.
        pattern_template (str): path pattern to dicom files, must have {subject} placeholder. Example: 'A:\\MR_Data\\Raw\\7T_Terra\\{subject}\\SCANS\\*\\DICOM\\*.dcm'
        sessions (list): List of session/subject identifiers to process. This will be replacing the subject placeholders in the pattern.
        force_heudiconv (bool): Whether to re-run even if outputs exist.
        dryrun (bool): If True, print commands without executing them.
        
    Raises:
        ValueError: If '{subject}' is not found in the pattern_template.
        
    """
        
    if "{subject}" not in pattern_template:
        raise ValueError("The given template does not contain the {subject} placeholder. Please specify with placeholder.")
    
    
    os.makedirs(output_dir, exist_ok=True)

    # Find previously processed sessions
    dicominfo_file_paths = glob.glob(
        r'**/dicominfo*.tsv',
        root_dir=output_dir,
        recursive=True,
        include_hidden=True
    )
    already_processed_sessions = [
        os.path.basename(os.path.dirname(os.path.dirname(p)))
        for p in dicominfo_file_paths
    ]

    print(f"Starting processing... \n")

    for i, session in enumerate(sessions):
       
        if session in already_processed_sessions and not force_heudiconv:
            print(f"=== SESSION ({i+1}/{len(sessions)}): {session:>35s} --> Already processed. Skipping.")
            continue
        else:
            print(f"==== SESSION ({i+1}/{len(sessions)}): {session:>35s} --> RUNNING HEUDICONV!")
            cmd = rf'heudiconv -d {pattern_template} -s {session} -o {output_dir} -c none -f convertall'

            print(f"Command: {cmd}")
            if not dryrun:
                process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, shell=True)

                # Stream stdout in real-time
                while True:
                    output = process.stdout.readline()
                    if output == "" and process.poll() is not None:
                        break
                    if output:
                        print(output.strip())
                        sys.stdout.flush()

                # Print any remaining stderr
                stderr_output = process.stderr.read()
                if stderr_output:
                    print(stderr_output.strip())
                    sys.stdout.flush()
            else:
                print("DRYRUN - don't run heudiconv (change flag to disable).")
            

    print(f"\n... success, all sessions processed!")


In [ ]:
# Load config
with open("CONFIG_FILE.json", "r") as f:
    cfg = json.load(f)

print("Loaded configuration:")
for k, v in cfg.items():
    print(f"{k}: '{v}'") 

# drive, _ = os.path.splitdrive(os.getcwd())
# print(f"\nYour current drive: '{drive}'")

anchor = Path.cwd()   # or notebook dir
root = src.utils.find_root_with_marker(anchor, ".drive_root.txt")

In [ ]:
#__MACOSX is a temp file from apple
IGNORE_FOLDERS = ["__MACOSX"] #ignore certain folders
IGNORE_HIDDEN = True #ignore hidden folders starting with a '.'. E.g. path/to/.thisisafoldertoignore

# Load in configs
dicom_root_path_7T = src.utils.combine_paths(root, cfg["RELATIVE_DICOM_ROOT_PATH_7T"])
dicom_root_path_3T = src.utils.combine_paths(root, cfg["RELATIVE_DICOM_ROOT_PATH_3T"])

output_dir_root_7T = src.utils.combine_paths(root, cfg["RELATIVE_OUTPUT_DIR_ROOT_7T"], check_if_exists=False)
output_dir_root_3T = src.utils.combine_paths(root, cfg["RELATIVE_OUTPUT_DIR_ROOT_3T"], check_if_exists=False)

folder_template_7T = cfg["GENERAL_FOLDER_TEMPLATE_7T"].format(dicom_root=dicom_root_path_7T, subject="{subject}") #paste in correct root path, subject stays as placeholder
folder_template_3T = cfg["GENERAL_FOLDER_TEMPLATE_3T"].format(dicom_root=dicom_root_path_3T, subject="{subject}") #paste in correct root path, subject stays as placeholder

dicominfo_filename_template = cfg["DICOMINFO_TSV_NAME"]

# 7T Data

In [ ]:
#not necessary to run this cell, just for info on the heudiconv tool :)
# !heudiconv --help

In [ ]:
#NB! for TRD7T0176_3 & TRD7T0176_5 patient_name is identical, hence foldername is different than patient_name 
#apart from those exceptions, please make sure to rename other folders such that they match the patient_name

subjects_7T = src.utils.extract_subject_foldername_and_patientid(pattern_template=folder_template_7T, 
                                    folders_to_ignore=IGNORE_FOLDERS, 
                                    ignore_hidden=True)
subjects_7T["patient_name"] = subjects_7T["patient_name"].apply(lambda x: x.replace("^", ""))
subjects_7T["is match?"] = subjects_7T.foldername == subjects_7T.patient_name


print("==========================================================")
print(f"Please rename folder according to DICOM patient_name.\
      \nNaming conflicts are an exception, please note them here.")
print("==========================================================")
print("Naming conflict for:\nTRD7T0176_3, TRD7T0176_5, TRD7T0387_2, TRD7T0176_3 --> same subject id, hence keep the suffix\n" \
"Phantom --> just adding date to the name so it is not confusing\n" \
"invivo_fmre_27_2_26 --> underscore necessary")

subjects_7T

In [ ]:
# 7T Terra

sessions_7T = subjects_7T.foldername.values.tolist()
sessions_7T


run_heudiconv_for_scanner(
    output_dir=output_dir_root_7T,
    pattern_template=folder_template_7T,
    sessions=sessions_7T,
    force_heudiconv=False,
    dryrun=False
)

# 3T data

In [ ]:
subjects_3T = src.utils.extract_subject_foldername_and_patientid(pattern_template=folder_template_3T, 
                                    folders_to_ignore=IGNORE_FOLDERS,
                                    ignore_hidden=True) 
subjects_3T["is match?"] = subjects_3T.foldername == subjects_3T.patient_name

print("==========================================================")
print(f"Please rename folder according to DICOM patient_name.\
      \nNaming conflicts are an exception, please note them here.")
print("==========================================================")
print("Naming conflict for:\nMRE_test_22/11/24 --> no slashes allowed in foldername, using underscores")

subjects_3T

In [ ]:
# 3T Skyra

sessions_3T = subjects_3T.foldername.values.tolist()
sessions_3T


run_heudiconv_for_scanner(
    output_dir=output_dir_root_3T,
    pattern_template=folder_template_3T,
    sessions=sessions_3T,
    force_heudiconv=False,
    dryrun=False
)

# Delete sensitive columns

we do not want e.g. patient age or sex in the overview sheets, thus delete! Overview sheets will be copied without named columns and renamed to specified TSV name.

In [ ]:
dicominfo_file_paths = []
for output_dir_root in [output_dir_root_7T, output_dir_root_3T]:
    cur_dicominfo_file_paths = glob.glob(r'**/dicominfo*.tsv', root_dir= output_dir_root, recursive= True, include_hidden=True)
    print(f"{len(cur_dicominfo_file_paths)} dicominfo files in {output_dir_root}")
    if cur_dicominfo_file_paths:
        cur_dicominfo_file_paths = [os.path.join(output_dir_root, p) for p in cur_dicominfo_file_paths]
        dicominfo_file_paths.append(cur_dicominfo_file_paths)

dicominfo_file_paths = [item for sublist in dicominfo_file_paths for item in sublist]  # Flatten the list of lists

print()
#to read out scan IDs of dicominfos (just for sanity check next cell)
dicominfo_scan_IDs = list()

columns_to_delete = ["patient_age", "patient_sex"]
for info_file_path in dicominfo_file_paths:
    if os.path.exists(info_file_path):
        subject_name = os.path.basename(os.path.dirname(os.path.dirname(info_file_path)))
        dicominfo_scan_IDs.append(subject_name)
        tsv_file = pd.read_csv(info_file_path, sep='\t', encoding='latin-1', on_bad_lines='skip')
        if any(c in tsv_file.columns for c in columns_to_delete):
            tsv_file.drop(columns=columns_to_delete, inplace=True, errors="ignore")
            # Write the modified DataFrame back to a new TSV file
            cur_dicominfo_filename = dicominfo_filename_template.format(subject = subject_name)
            new_file_path = os.path.join(os.path.dirname(info_file_path), cur_dicominfo_filename)
            tsv_file.to_csv(new_file_path, sep='\t', index=False)
            print(f"deleted sensitive columns for {info_file_path}")
            os.remove(info_file_path)
        else:
            print(f"No sensitive columns found for {info_file_path}")

# Check for missing data or missing info sheet
do we have info sheets for all data, and data for all info sheets?

In [ ]:
sessions = sessions_7T + sessions_3T
if sorted(sessions) != sorted(dicominfo_scan_IDs):
    no_info = list(set(sessions) - set(dicominfo_scan_IDs))
    no_data = list(set(dicominfo_scan_IDs) - set(sessions))
    print(f"Existing scans that have no dicominfo.tsv: {no_info}")
    print(f"Existing dicominfo.tsv where no scan is found: {no_data}")
    raise ValueError("Found mismatch in found Scan data and existing summary sheets. See print output above.")
else:
    print("For all scans a dicominfo sheet was found, and for all dicominfo sheets a scan was found.")